In [24]:
pip install google-cloud-bigquery pandas pyarrow db-dtypes requests 

In [25]:
import os

print("Project ID found:", bool(os.environ.get("gcl_project_id")))
print("GCP service account key found:", bool(os.environ.get("GCP_SA_KEY")))

Project ID found: True
GCP service account key found: True


In [26]:
import sys, json
import pandas as pd
from google.cloud import bigquery

PROJECT_ID = os.environ["gcl_project_id"]   # ← замінити на свій проєкт
LOCATION   = "EU"

creds = None

if "google.colab" in sys.modules:                  # Colab
    from google.colab import auth
    auth.authenticate_user()

elif os.environ.get("GCP_SA_KEY"):                 # Datalore: ключ у змінній середовища
    from google.oauth2 import service_account
    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"])

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)
print("проєкт:", client.project)

проєкт: datalab-504011


In [28]:
import requests
import hashlib
import pandas as pd
from datetime import date, datetime, timezone

# Завдання 2.1. Створіть датасети nbu_raw і nbu_dwh у регіоні EU

ds = bigquery.Dataset(f"{PROJECT_ID}.nbu_raw")
ds.location = "EU"
client.create_dataset(ds, exists_ok=True)

ds = bigquery.Dataset(f"{PROJECT_ID}.nbu_dwh")
ds.location = "EU"
client.create_dataset(ds, exists_ok=True)

# Завдання 2.2. Створіть таблицю

rates_table = f"{PROJECT_ID}.nbu_raw.raw_rates"

schema = [
    bigquery.SchemaField("ingested_at", "TIMESTAMP"),
    bigquery.SchemaField("business_date", "DATE"),
    bigquery.SchemaField("request_url", "STRING"),
    bigquery.SchemaField("payload", "STRING"),
    bigquery.SchemaField("payload_hash", "STRING"),
]

table = bigquery.Table(rates_table, schema=schema)
table.time_partitioning = bigquery.TimePartitioning(field="business_date")

table = client.create_table(table, exists_ok=True)

print("Created", table.full_table_id)

#Завдання 2.3. Заберіть із API курс на сьогодні та виведіть кількість отриманих валют і перший елемент відповіді


URL = "https://bank.gov.ua/NBUStatService/v1/statdirectory/exchange?json"

resp = requests.get(URL, timeout=30)
resp.raise_for_status()
data = resp.json()          # список словників

print(len(data), "валют")
print(data[0])

# 2.4. Перетворіть відповідь на рядки таблиці:

business_date = date.today()

rows = []

for item in data:

    # Entire API object as JSON text
    payload = json.dumps(
        item,
        ensure_ascii=False,
        sort_keys=True
    )

    # MD5 of that JSON text
    payload_hash = hashlib.md5(
        payload.encode("utf-8")
    ).hexdigest()

    rows.append({
        "ingested_at": datetime.now(timezone.utc),
        "business_date": business_date,
        "request_url": URL,
        "payload": payload,
        "payload_hash": payload_hash
    })

df = pd.DataFrame(rows)
df

# Завдання 2.5. Запишіть DataFrame у nbu_raw.raw_rates у режимі WRITE_APPEND.

cfg = bigquery.LoadJobConfig(schema=schema,
                              write_disposition="WRITE_APPEND")
client.load_table_from_dataframe(df, rates_table, job_config=cfg).result()

# Завдання 2.6. Виконайте перевірочний запит і виведіть кількість рядків за кожною business_date

query_check = f"""
SELECT business_date, COUNT(*) AS rows_cnt, COUNT(DISTINCT payload_hash) AS uniq_cnt
FROM `{rates_table}`
GROUP BY business_date
ORDER BY business_date
"""

df_check = client.query(query_check).to_dataframe()

df_check

# Завдання 2.7. Запустіть ноутбук двічі підряд і подивіться на результат запиту із пункту 2.6:

# Так відбувається тому що payload_hash завжди має унікальне значеня яке базується на унікальності колонки payload 
# і ніколи не дублюється

Created datalab-504011:nbu_raw.raw_rates
45 валют
{'r030': 12, 'txt': 'Алжирський динар', 'rate': 0.33432, 'cc': 'DZD', 'exchangedate': '01.09.2026', 'special': None}


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(
/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,business_date,rows_cnt,uniq_cnt
0,2026-08-24,90,45
1,2026-08-27,45,45
2,2026-08-28,90,45
3,2026-08-29,90,45
4,2026-08-31,360,45


In [27]:
import requests

URL = "https://bank.gov.ua/NBUStatService/v1/statdirectory/exchange?json"

resp = requests.get(URL, timeout=30)
resp.raise_for_status()
data = resp.json()          # список словників

print(len(data), "валют")
print(data[0])

45 валют
{'r030': 12, 'txt': 'Алжирський динар', 'rate': 0.33432, 'cc': 'DZD', 'exchangedate': '01.09.2026', 'special': None}
